# Ecuación de calor 2D en placa: cuaderno guiado

Modelo:

$$
\frac{\partial u}{\partial t} = \alpha(u_{xx}+u_{yy}),
\qquad (x,y)\in(0,1)^2.
$$

Fronteras del ejemplo:

- $u(0,y,t)=0$,
- $u(x,0,t)=0$,
- $u(1,y,t)=1$,
- $u(x,1,t)=1$.

Condición inicial:

$$
 u(x,y,0)=0.25\exp\!\left(-\frac{(x-0.35)^2+(y-0.35)^2}{0.02}\right).
$$


## Esquema explícito

$$
u_{i,j}^{n+1}=u_{i,j}^n+r\big(u_{i+1,j}^n+u_{i-1,j}^n+u_{i,j+1}^n+u_{i,j-1}^n-4u_{i,j}^n\big),
\quad r=\frac{\alpha\Delta t}{h^2}.
$$

Condición de estabilidad para malla isotrópica: $r\le 1/4$.


## Estabilidad (derivación corta)

Con malla general:

$$
r_x=\frac{\alpha\Delta t}{\Delta x^2},\qquad
r_y=\frac{\alpha\Delta t}{\Delta y^2}.
$$

El análisis modal de Fourier conduce a:

$$
G=1-4r_x\sin^2\!\left(\frac{\theta_x}{2}\right)-4r_y\sin^2\!\left(\frac{\theta_y}{2}\right).
$$

Para imponer $|G|\le 1$ en todos los modos:

$$
r_x+r_y\le \frac12.
$$

Si $\Delta x=\Delta y=h$, entonces $r_x=r_y=r$ y recuperamos:

$$
r\le \frac14.
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

alpha = 1.0
L = 1.0
T = 0.5
N = 34
r_obj = 0.22


In [ ]:
def apply_bc(u):
    u[:, 0] = 0.0
    u[0, :] = 0.0
    u[:, -1] = 1.0
    u[-1, :] = 1.0
    return u


def build_problem(alpha=1.0, L=1.0, T=0.5, N=34, r_obj=0.22):
    h = L / N
    dt = r_obj * h * h / alpha
    nt = int(np.ceil(T / dt))
    dt = T / nt
    r = alpha * dt / (h * h)
    x = np.linspace(0.0, L, N + 1)
    y = np.linspace(0.0, L, N + 1)
    X, Y = np.meshgrid(x, y, indexing="xy")
    u = 0.25 * np.exp(-((X - 0.35)**2 + (Y - 0.35)**2) / 0.02)
    u = apply_bc(u)
    return x, y, dt, nt, r, u


In [ ]:
def step_heat2d(u, r):
    un = u.copy()
    u_new = un.copy()
    u_new[1:-1, 1:-1] = un[1:-1, 1:-1] + r * (
        un[2:, 1:-1] + un[:-2, 1:-1] + un[1:-1, 2:] + un[1:-1, :-2] - 4.0 * un[1:-1, 1:-1]
    )
    u_new = apply_bc(u_new)
    return u_new


def solve_heat2d(alpha=1.0, L=1.0, T=0.5, N=34, r_obj=0.22, store_every=4):
    x, y, dt, nt, r, u = build_problem(alpha=alpha, L=L, T=T, N=N, r_obj=r_obj)
    t_hist = [0.0]
    U_hist = [u.copy()]
    for n in range(nt):
        u = step_heat2d(u, r)
        t_next = (n + 1) * dt
        if (n + 1) % store_every == 0 or (n + 1) == nt:
            t_hist.append(t_next)
            U_hist.append(u.copy())
    return x, y, r, np.array(t_hist), np.array(U_hist)


x, y, r, t_hist, U_hist = solve_heat2d(alpha=alpha, L=L, T=T, N=N, r_obj=r_obj, store_every=4)
print(f"r={r:.6f}")
print(f"t_final={t_hist[-1]:.6f}")


In [ ]:
plt.figure(figsize=(6.6, 5.3))
im = plt.imshow(U_hist[-1], extent=[0, 1, 0, 1], origin="lower", cmap="turbo", vmin=0.0, vmax=1.0, aspect="auto")
plt.colorbar(im, orientation="horizontal", pad=0.12, label="Temperatura")
plt.title("Calor 2D en t_final")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()
